In [7]:
from __future__ import annotations

import re
import sqlite3
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [8]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "nepse_analyst").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from nepse_analyst.database import create_database
from nepse_analyst.config import DB_PATH

create_database()

SHARESANSAR_EXISTING_ISSUES = "https://www.sharesansar.com/existing-issues"
SHARESANSAR_IPO_RESULT = "https://www.sharesansar.com/ipo-result"
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
)

In [9]:
def to_float(value: object) -> float | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    text = text.replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else None


def to_int(value: object) -> int | None:
    parsed = to_float(value)
    if parsed is None:
        return None
    return int(parsed)


def normalize_company_name(value: str) -> str:
    text = value.lower()
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text).strip()
    return re.sub(r"\s+", " ", text)


def ensure_ipos_columns(conn: sqlite3.Connection) -> None:
    existing = {row[1] for row in conn.execute("PRAGMA table_info(ipos)")}
    if "oversubscription_rate" not in existing:
        conn.execute("ALTER TABLE ipos ADD COLUMN oversubscription_rate REAL")
        conn.commit()


def first_trade_map(conn: sqlite3.Connection) -> pd.DataFrame:
    query = """
    WITH first_day AS (
        SELECT symbol, MIN(trade_date) AS first_trade_date
        FROM price_history
        WHERE symbol IS NOT NULL
        GROUP BY symbol
    )
    SELECT
        f.symbol,
        f.first_trade_date AS listing_date_from_market,
        COALESCE(p.open_price, p.close_price) AS listing_price_from_market
    FROM first_day f
    JOIN price_history p
      ON p.symbol = f.symbol
     AND p.trade_date = f.first_trade_date
    """
    return pd.read_sql_query(query, conn)


def companies_map(conn: sqlite3.Connection) -> pd.DataFrame:
    return pd.read_sql_query(
        "SELECT symbol, name AS company_name_db, sector FROM companies",
        conn,
    )

In [10]:
def fetch_existing_issues_rows(session: requests.Session) -> list[dict]:
    response = session.get(SHARESANSAR_EXISTING_ISSUES, timeout=30)
    response.raise_for_status()

    keys = [
        "DT_Row_Index",
        "company.symbol",
        "company.companyname",
        "ratio_value",
        "total_units",
        "issue_price",
        "price_range",
        "cutoff_price",
        "opening_date",
        "closing_date",
        "final_date",
        "listing_date",
        "issue_manager",
        "status",
        "view",
        "right_eligibility_link",
    ]

    params: dict[str, str] = {
        "type": "1",
        "draw": "1",
        "start": "0",
        "length": "500",
        "search[value]": "",
        "search[regex]": "false",
        "order[0][column]": "0",
        "order[0][dir]": "asc",
    }
    for idx, key in enumerate(keys):
        params[f"columns[{idx}][data]"] = key
        params[f"columns[{idx}][name]"] = ""
        params[f"columns[{idx}][searchable]"] = "true"
        params[f"columns[{idx}][orderable]"] = "false"
        params[f"columns[{idx}][search][value]"] = ""
        params[f"columns[{idx}][search][regex]"] = "false"

    json_response = session.get(
        SHARESANSAR_EXISTING_ISSUES,
        params=params,
        headers={
            "X-Requested-With": "XMLHttpRequest",
            "Accept": "application/json, text/javascript, */*; q=0.01",
        },
        timeout=30,
    )

    rows = []
    try:
        payload = json_response.json()
        rows = payload.get("data") or []
    except Exception:
        rows = []

    normalized: list[dict] = []
    for row in rows:
        company = row.get("company") or {}
        status_code = row.get("status")
        listing_date = str(row.get("listing_date") or "").strip()

        oversub_ratio = to_float(row.get("ratio_value"))
        normalized.append(
            {
                "company_name": str(company.get("companyname") or "").strip(),
                "symbol": str(company.get("symbol") or "").strip().upper(),
                "issue_date": str(row.get("opening_date") or "").strip() or None,
                "listing_date": listing_date or None,
                "issue_price": to_float(row.get("issue_price")),
                "shares_issued": to_int(row.get("total_units")),
                "total_applicants": None,
                "allotment_per_person": None,
                "oversubscription_rate": oversub_ratio,
                "status": "listed" if status_code not in (-2, -1, 0) and listing_date else "open",
            }
        )

    return normalized


def fetch_ipo_result_option_names(session: requests.Session) -> list[str]:
    page = session.get(SHARESANSAR_IPO_RESULT, timeout=30)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "lxml")

    names: list[str] = []
    select_tag = soup.select_one("select#companyid")
    if not select_tag:
        return names

    for option in select_tag.select("option"):
        name = option.get_text(strip=True)
        if name:
            names.append(name)

    return names

In [11]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

with sqlite3.connect(DB_PATH) as conn:
    ensure_ipos_columns(conn)
    companies_df = companies_map(conn)
    first_trade_df = first_trade_map(conn)

raw_rows = fetch_existing_issues_rows(session)

if not raw_rows:
    option_names = fetch_ipo_result_option_names(session)
    raw_rows = [{
        "company_name": name,
        "symbol": None,
        "issue_date": None,
        "listing_date": None,
        "issue_price": 100.0,
        "shares_issued": None,
        "total_applicants": None,
        "allotment_per_person": None,
        "oversubscription_rate": None,
        "status": "listed",
    } for name in option_names]

ipo_df = pd.DataFrame(raw_rows)

company_lookup = companies_df.copy()
company_lookup["company_name_norm"] = company_lookup["company_name_db"].map(normalize_company_name)
ipo_df["company_name_norm"] = ipo_df["company_name"].map(normalize_company_name)

ipo_df = ipo_df.merge(
    company_lookup[["symbol", "company_name_norm", "sector"]],
    how="left",
    on="company_name_norm",
    suffixes=("", "_company"),
)

ipo_df["symbol"] = ipo_df["symbol"].fillna(ipo_df.get("symbol_company"))
if "symbol_company" in ipo_df.columns:
    ipo_df = ipo_df.drop(columns=["symbol_company"])

ipo_df = ipo_df.merge(first_trade_df, how="left", on="symbol")

ipo_df["listing_date"] = ipo_df["listing_date"].fillna(ipo_df["listing_date_from_market"])
ipo_df["listing_price"] = ipo_df["listing_price_from_market"]

derived_mask = ipo_df["total_applicants"].notna() & ipo_df["shares_issued"].notna() & ipo_df["allotment_per_person"].notna()
ipo_df.loc[derived_mask, "oversubscription_rate"] = (
    ipo_df.loc[derived_mask, "total_applicants"]
    / (ipo_df.loc[derived_mask, "shares_issued"] / ipo_df.loc[derived_mask, "allotment_per_person"])
)

ipo_df["status"] = "listed"
ipo_df = ipo_df[ipo_df["symbol"].notna()]
ipo_df = ipo_df[ipo_df["listing_price"].notna()]

final_columns = [
    "symbol",
    "company_name",
    "sector",
    "issue_date",
    "listing_date",
    "issue_price",
    "listing_price",
    "shares_issued",
    "total_applicants",
    "oversubscription_rate",
    "status",
]

final_df = ipo_df[final_columns].drop_duplicates(subset=["symbol"], keep="first")

with sqlite3.connect(DB_PATH) as conn:
    ensure_ipos_columns(conn)
    conn.execute("DELETE FROM ipos")
    final_df.to_sql("ipos", conn, if_exists="append", index=False)

print(f"Loaded IPO rows: {len(final_df)}")
print("Listing price source: first available trading-day price in price_history (open_price fallback close_price).")
final_df.head(10)

Loaded IPO rows: 15
Listing price source: first available trading-day price in price_history (open_price fallback close_price).


,symbol,company_name,sector,issue_date,listing_date,issue_price,listing_price,shares_issued,total_applicants,oversubscription_rate,status
0,GLH,Greenlife Hydropower Limited,Hydropower,None,2021-04-25,100.0,379.0,None,None,None,listed
1,NIFRA,Nepal Infrastructure Bank Limited,Investment,None,2021-04-25,100.0,565.0,None,None,None,listed
2,CGH,Chandragiri Hills Limited,Hotels & Tourism,None,2021-02-07,100.0,161.0,None,None,None,listed
7,MEN,Mountain Energy Nepal Limited,Hydropower,None,2021-02-07,100.0,665.0,None,None,None,listed
12,LEC,Liberty Energy Company Limited,Hydropower,None,2020-11-26,100.0,204.0,None,None,None,listed
17,NRN,NRN Infrastructure and Development Limited,Investment,None,2020-09-09,100.0,374.0,None,None,None,listed
25,HDHPC,Himal Dolakha Hydropower Company Limited,Hydropower,None,2020-03-12,100.0,96.0,None,None,None,listed
35,RHPL,Rasuwagadhi Hydropower Company Limited (IPO),Hydropower,None,2019-08-11,100.0,210.0,None,None,None,listed
39,HURJA,Himalaya Urja Bikas Company Limited,Hydropower,None,2019-05-22,100.0,160.0,None,None,None,listed
42,MHNL,Mountain Hydro Nepal Limited,Hydropower,None,2019-05-12,100.0,91.0,None,None,None,listed


In [12]:
with sqlite3.connect(DB_PATH) as conn:
    verification = pd.read_sql_query(
        """
        SELECT
            company_name,
            issue_price,
            listing_price,
            (listing_price - issue_price) / issue_price * 100 AS listing_premium_pct
        FROM ipos
        WHERE issue_price IS NOT NULL AND issue_price > 0
        ORDER BY listing_premium_pct DESC
        LIMIT 10
        """,
        conn,
    )

verification

,company_name,issue_price,listing_price,listing_premium_pct
0,Nepal Bank Limited,100.0,900.0,800.0
1,Mountain Energy Nepal Limited,100.0,665.0,565.0
2,Nepal Infrastructure Bank Limited,100.0,565.0,465.0
3,BUTWAL POWER COMPANY LIMITED,100.0,480.0,380.0
4,Greenlife Hydropower Limited,100.0,379.0,279.0
5,NRN Infrastructure and Development Limited,100.0,374.0,274.0
6,NMB Bank Limited,100.0,228.0,128.0
7,Rasuwagadhi Hydropower Company Limited (IPO),100.0,210.0,110.0
8,Liberty Energy Company Limited,100.0,204.0,104.0
9,Chandragiri Hills Limited,100.0,161.0,61.0
